# ⚙️ Phase 2 — Prétraitement & DataLoaders
**Projet : Détection d'Anomalies dans des Objets Industriels 3D — Catégorie : bagel**

---
**Objectifs de cette phase :**
- Nettoyer et normaliser les nuages de points 3D
- Sous-échantillonner à taille fixe (N points)
- Augmenter les données (rotation, bruit, jitter)
- Construire les DataLoaders PyTorch pour l'entraînement
- Vérifier et visualiser le pipeline complet

**Résultats Phase 1 — bagel :**
- Train : 244 pièces normales | Validation : 22 | Test : 110
- Défauts : combined, contamination, crack, hole
- Format 3D : TIFF (H×W×3), plage Z ≈ [0.52, 0.60]

## 1. Installation et imports

In [ ]:
!pip install open3d tifffile torch torchvision matplotlib numpy -q
print('✅ Dépendances installées')

In [ ]:
import os
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split

# ─── Chemins (adaptez si nécessaire) ──────────────────────────────────────
EXTRACT_DIR = '/content/mvtec3d'
CATEGORY    = 'bagel'
CAT_PATH    = Path(EXTRACT_DIR) / CATEGORY

# ─── Hyperparamètres de prétraitement ─────────────────────────────────────
N_POINTS    = 2048    # nombre de points après sous-échantillonnage
BATCH_SIZE  = 16      # taille des batchs pour l'entraînement
SEED        = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'✅ Config : N_POINTS={N_POINTS}, BATCH_SIZE={BATCH_SIZE}')

## 2. Fonctions de chargement des données brutes

In [ ]:
def load_xyz_tiff(path):
    """
    Charge un fichier .tiff MVTec 3D-AD.
    Chaque pixel (H, W) contient (X, Y, Z).
    Retourne un tableau (N, 3) de points valides uniquement.
    """
    xyz_map = tifffile.imread(str(path))        # (H, W, 3)
    mask    = np.any(xyz_map != 0, axis=-1)     # masque des pixels valides
    points  = xyz_map[mask].astype(np.float32)  # (N, 3)
    return points


def load_gt_mask(path):
    """
    Charge un masque GT (ground truth).
    Retourne un tableau binaire (H, W) : 1 = anomalie, 0 = normal.
    """
    mask = plt.imread(str(path))
    if mask.ndim == 3:
        mask = mask[:, :, 0]     # garder un seul canal
    return (mask > 0.5).astype(np.float32)


# Test rapide
sample_xyz = sorted((CAT_PATH / 'train' / 'good' / 'xyz').glob('*.tiff'))[0]
pts = load_xyz_tiff(sample_xyz)
print(f'✅ Chargement OK : {pts.shape[0]:,} points bruts pour le premier bagel')
print(f'   Plage X : [{pts[:,0].min():.4f}, {pts[:,0].max():.4f}]')
print(f'   Plage Y : [{pts[:,1].min():.4f}, {pts[:,1].max():.4f}]')
print(f'   Plage Z : [{pts[:,2].min():.4f}, {pts[:,2].max():.4f}]')

## 3. Prétraitement — Normalisation et recentrage

In [ ]:
def normalize_pointcloud(points):
    """
    Normalisation standard d'un nuage de points :
    1. Recentrage sur le barycentre (centroid = 0)
    2. Mise à l'échelle dans une sphère unitaire (max distance = 1)

    Cette normalisation rend le modèle invariant à la position
    et à l'échelle absolue de la pièce.
    """
    # Étape 1 : recentrage
    centroid = points.mean(axis=0)       # barycentre (3,)
    points   = points - centroid         # centrage

    # Étape 2 : normalisation à l'échelle unitaire
    max_dist = np.max(np.linalg.norm(points, axis=1))  # distance max au centre
    points   = points / max_dist         # tous les points dans [-1, 1]

    return points, centroid, max_dist


# Démonstration sur un bagel
pts_raw  = load_xyz_tiff(sample_xyz)
pts_norm, centroid, scale = normalize_pointcloud(pts_raw)

print('=== Avant normalisation ===')
print(f'  Centroïde : {pts_raw.mean(axis=0).round(4)}')
print(f'  Distance max au centre : {np.max(np.linalg.norm(pts_raw - pts_raw.mean(axis=0), axis=1)):.4f}')

print('\n=== Après normalisation ===')
print(f'  Centroïde : {pts_norm.mean(axis=0).round(6)}')
print(f'  Distance max au centre : {np.max(np.linalg.norm(pts_norm, axis=1)):.4f}  (doit être 1.0)')
print(f'  Plage Z normalisée : [{pts_norm[:,2].min():.4f}, {pts_norm[:,2].max():.4f}]')

In [ ]:
# Visualisation avant / après normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, pts, title in zip(axes,
                           [pts_raw, pts_norm],
                           ['Brut (coordonnées réelles)', 'Normalisé (sphère unitaire)']):
    sc = ax.scatter(pts[:2000, 0], pts[:2000, 1],
                    c=pts[:2000, 2], cmap='viridis', s=1, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Z')
    ax.set_title(title)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_aspect('equal')

plt.suptitle('bagel — Nuage de points : vue de dessus (projection XY)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Prétraitement — Sous-échantillonnage à N points fixes

In [ ]:
def random_sample(points, n_points=2048):
    """
    Sous-échantillonnage aléatoire uniforme.
    Si le nuage a moins de n_points, on échantillonne avec remplacement.
    Résultat : tableau de forme fixe (n_points, 3).
    """
    n = len(points)
    replace = n < n_points
    idx = np.random.choice(n, n_points, replace=replace)
    return points[idx]


def fps_sample(points, n_points=2048):
    """
    Farthest Point Sampling (FPS) — sous-échantillonnage uniforme de qualité.
    Sélectionne les points les plus éloignés les uns des autres.
    Plus lent que random mais meilleure couverture spatiale.
    """
    n = len(points)
    selected = np.zeros(n_points, dtype=int)
    distances = np.full(n, np.inf)

    # Point de départ aléatoire
    selected[0] = np.random.randint(0, n)

    for i in range(1, n_points):
        last = points[selected[i - 1]]
        dist = np.sum((points - last) ** 2, axis=1)
        distances = np.minimum(distances, dist)
        selected[i] = np.argmax(distances)

    return points[selected]


# Comparaison Random vs FPS
pts_norm_full = pts_norm.copy()

pts_random = random_sample(pts_norm_full, N_POINTS)
print(f'✅ Random sampling : {pts_random.shape}')

# FPS sur un sous-ensemble pour la rapidité en démo
pts_fps = fps_sample(pts_norm_full[:5000], N_POINTS)
print(f'✅ FPS sampling    : {pts_fps.shape}')

In [ ]:
# Visualisation Random vs FPS
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pts, title in zip(axes,
                           [pts_random, pts_fps],
                           [f'Random sampling ({N_POINTS} pts)', f'FPS sampling ({N_POINTS} pts)']):
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=pts[:, 2],
                    cmap='plasma', s=3, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='Z')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_aspect('equal')

plt.suptitle('Comparaison des stratégies de sous-échantillonnage — bagel', fontsize=12)
plt.tight_layout()
plt.show()
print('💡 FPS = meilleure couverture spatiale (préféré pour PointNet)')
print('💡 Random = plus rapide (acceptable pour autoencodeur)')

## 5. Prétraitement — Augmentation de données

In [ ]:
def augment_pointcloud(points, training=True):
    """
    Augmentations appliquées uniquement pendant l'entraînement.
    Objectif : augmenter la variabilité artificielle pour améliorer
    la robustesse du modèle face aux variations naturelles des pièces.

    Augmentations appliquées :
    1. Rotation aléatoire autour de l'axe Z (invariance rotationnelle)
    2. Jitter gaussien (bruit de capteur)
    3. Mise à l'échelle aléatoire (variation de taille naturelle)
    """
    if not training:
        return points

    pts = points.copy()

    # 1. Rotation aléatoire autour de Z (les bagels peuvent être orientés différemment)
    theta = np.random.uniform(0, 2 * np.pi)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    R = np.array([[cos_t, -sin_t, 0],
                  [sin_t,  cos_t, 0],
                  [0,      0,     1]], dtype=np.float32)
    pts = pts @ R.T

    # 2. Jitter gaussien (simule le bruit du scanner)
    noise = np.random.normal(0, 0.005, size=pts.shape).astype(np.float32)
    pts   = pts + noise

    # 3. Mise à l'échelle aléatoire légère (±10%)
    scale = np.random.uniform(0.9, 1.1)
    pts   = pts * scale

    return pts.astype(np.float32)


# Démonstration des augmentations
pts_base = random_sample(pts_norm, N_POINTS)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Aug. 1', 'Aug. 2', 'Aug. 3']

axes[0].scatter(pts_base[:, 0], pts_base[:, 1],
                c=pts_base[:, 2], cmap='viridis', s=2, alpha=0.6)
axes[0].set_title('Original', fontsize=10)
axes[0].set_aspect('equal')
axes[0].axis('off')

for i in range(1, 4):
    pts_aug = augment_pointcloud(pts_base, training=True)
    axes[i].scatter(pts_aug[:, 0], pts_aug[:, 1],
                    c=pts_aug[:, 2], cmap='viridis', s=2, alpha=0.6)
    axes[i].set_title(f'Augmenté #{i}', fontsize=10)
    axes[i].set_aspect('equal')
    axes[i].axis('off')

plt.suptitle('Augmentations de données (rotation + jitter + scale) — bagel', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Construction du Dataset PyTorch

In [ ]:
def collect_samples(cat_path, split):
    """
    Collecte tous les chemins de fichiers pour un split donné.
    Retourne une liste de dictionnaires :
      {'xyz': path, 'label': 0/1, 'defect_type': str, 'gt': path_or_None}
    """
    samples = []
    split_path = cat_path / split

    if not split_path.exists():
        print(f'⚠️ Split {split} introuvable')
        return samples

    for defect_dir in sorted(split_path.iterdir()):
        if not defect_dir.is_dir():
            continue

        is_good = (defect_dir.name == 'good')
        label   = 0 if is_good else 1
        xyz_dir = defect_dir / 'xyz'
        gt_dir  = defect_dir / 'gt'

        if not xyz_dir.exists():
            continue

        for xyz_file in sorted(xyz_dir.glob('*.tiff')):
            # Chercher le masque GT correspondant
            gt_file = None
            if not is_good and gt_dir.exists():
                stem = xyz_file.stem
                candidates = list(gt_dir.glob(f'{stem}*.png'))
                if candidates:
                    gt_file = candidates[0]

            samples.append({
                'xyz':         xyz_file,
                'label':       label,
                'defect_type': defect_dir.name,
                'gt':          gt_file
            })

    return samples


# Collecte pour chaque split
train_samples = collect_samples(CAT_PATH, 'train')
val_samples   = collect_samples(CAT_PATH, 'validation')
test_samples  = collect_samples(CAT_PATH, 'test')

print(f'✅ Échantillons collectés :')
print(f'   Train      : {len(train_samples)} (toutes normales)')
print(f'   Validation : {len(val_samples)}')
print(f'   Test       : {len(test_samples)}')

In [ ]:
class BagelDataset(Dataset):
    """
    Dataset PyTorch pour MVTec 3D-AD — catégorie bagel.

    Pipeline appliqué à chaque pièce :
    1. Chargement du .tiff XYZ
    2. Filtrage des points invalides
    3. Normalisation (centrage + sphère unitaire)
    4. Sous-échantillonnage à N_POINTS
    5. Augmentation (train uniquement)
    6. Conversion en tenseur PyTorch (float32)

    Retourne :
    - points  : (N_POINTS, 3) float32
    - label   : 0 (normal) ou 1 (défectueux)
    - defect  : nom du type de défaut (str)
    """

    def __init__(self, samples, n_points=2048, training=False, augment=False):
        self.samples   = samples
        self.n_points  = n_points
        self.training  = training
        self.augment   = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # 1. Chargement
        points = load_xyz_tiff(sample['xyz'])

        # 2. Normalisation
        points, _, _ = normalize_pointcloud(points)

        # 3. Sous-échantillonnage
        points = random_sample(points, self.n_points)

        # 4. Augmentation (train uniquement)
        if self.augment and self.training:
            points = augment_pointcloud(points, training=True)

        # 5. Conversion tenseur
        points_tensor = torch.from_numpy(points).float()   # (N, 3)
        label_tensor  = torch.tensor(sample['label']).long()

        return points_tensor, label_tensor, sample['defect_type']


# Création des datasets
train_dataset = BagelDataset(train_samples, n_points=N_POINTS,
                              training=True, augment=True)
val_dataset   = BagelDataset(val_samples,   n_points=N_POINTS,
                              training=False, augment=False)
test_dataset  = BagelDataset(test_samples,  n_points=N_POINTS,
                              training=False, augment=False)

print(f'✅ Datasets créés :')
print(f'   Train  : {len(train_dataset)} échantillons')
print(f'   Val    : {len(val_dataset)} échantillons')
print(f'   Test   : {len(test_dataset)} échantillons')

# Test d'un item
pts_t, lbl_t, def_t = train_dataset[0]
print(f'\n   Shape tenseur  : {pts_t.shape}  (attendu : [{N_POINTS}, 3])')
print(f'   dtype          : {pts_t.dtype}')
print(f'   Label          : {lbl_t.item()} ({def_t})')
print(f'   Plage valeurs  : [{pts_t.min():.4f}, {pts_t.max():.4f}]')

## 7. Construction des DataLoaders PyTorch

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # mélange à chaque époque
    num_workers=2,         # chargement parallèle
    pin_memory=True,       # transfert GPU plus rapide
    drop_last=True         # évite les batchs incomplets en fin d'époque
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print('✅ DataLoaders créés')
print(f'   Train  : {len(train_loader)} batchs × {BATCH_SIZE} = {len(train_loader)*BATCH_SIZE} échantillons')
print(f'   Val    : {len(val_loader)} batchs')
print(f'   Test   : {len(test_loader)} batchs')

In [ ]:
# ── Vérification d'un batch complet ──────────────────────────────────────
batch_pts, batch_labels, batch_defects = next(iter(train_loader))

print('=== Vérification du batch train ===')
print(f'  batch_pts shape   : {batch_pts.shape}   (attendu : [{BATCH_SIZE}, {N_POINTS}, 3])')
print(f'  batch_labels      : {batch_labels.tolist()}')
print(f'  batch_defects     : {batch_defects}')
print(f'  Plage globale     : [{batch_pts.min():.4f}, {batch_pts.max():.4f}]')
print(f'  Pas de NaN        : {not torch.isnan(batch_pts).any()}')
print(f'  Pas de Inf        : {not torch.isinf(batch_pts).any()}')

## 8. Visualisation du pipeline complet

In [ ]:
# Comparaison : pièce normale vs pièce défectueuse après pipeline complet
def get_sample_by_type(dataset, defect_type):
    """Récupère le premier échantillon d'un type donné."""
    for i, s in enumerate(dataset.samples):
        if s['defect_type'] == defect_type:
            return dataset[i]
    return None


# Récupérer good, crack, hole, contamination du test
types_to_show = ['good', 'crack', 'hole', 'contamination', 'combined']
results = {t: get_sample_by_type(test_dataset, t) for t in types_to_show}

fig, axes = plt.subplots(2, len(types_to_show), figsize=(4 * len(types_to_show), 8))

for col, (defect, result) in enumerate(results.items()):
    if result is None:
        continue
    pts, label, _ = result
    pts_np = pts.numpy()

    # Vue de dessus (XY)
    ax_top = axes[0][col]
    ax_top.scatter(pts_np[:, 0], pts_np[:, 1],
                   c=pts_np[:, 2], cmap='viridis', s=2, alpha=0.7)
    color = 'green' if defect == 'good' else 'red'
    ax_top.set_title(defect, color=color, fontsize=10, fontweight='bold')
    ax_top.set_xlabel('X'); ax_top.set_ylabel('Y')
    ax_top.set_aspect('equal')

    # Vue de côté (XZ)
    ax_side = axes[1][col]
    ax_side.scatter(pts_np[:, 0], pts_np[:, 2],
                    c=pts_np[:, 1], cmap='plasma', s=2, alpha=0.7)
    ax_side.set_xlabel('X'); ax_side.set_ylabel('Z')
    ax_side.set_aspect('equal')

axes[0][0].set_ylabel('Vue dessus (XY)', fontsize=9)
axes[1][0].set_ylabel('Vue côté (XZ)', fontsize=9)

plt.suptitle(f'Pipeline complet — {N_POINTS} points normalisés par pièce — bagel', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Sauvegarde du pipeline prétraité

In [ ]:
import pickle

# ─── Sauvegarder les listes de samples pour réutilisation rapide ──────────
SAVE_DIR = '/content/drive/MyDrive/Detection_Anomalie_3D/preprocessed'
os.makedirs(SAVE_DIR, exist_ok=True)

# Convertir les Path en str pour la sérialisation
def serialize_samples(samples):
    return [{'xyz': str(s['xyz']), 'label': s['label'],
             'defect_type': s['defect_type'],
             'gt': str(s['gt']) if s['gt'] else None}
            for s in samples]

config = {
    'category':     CATEGORY,
    'n_points':     N_POINTS,
    'batch_size':   BATCH_SIZE,
    'seed':         SEED,
    'train_samples': serialize_samples(train_samples),
    'val_samples':   serialize_samples(val_samples),
    'test_samples':  serialize_samples(test_samples),
}

save_path = os.path.join(SAVE_DIR, f'{CATEGORY}_config.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(config, f)

print(f'✅ Configuration sauvegardée : {save_path}')
print(f'   → Réutilisable en Phase 3 sans réextraire le dataset')

## 10. Bilan Phase 2 & Décisions pour la Phase 3

In [ ]:
print('=' * 60)
print(f'  BILAN PHASE 2 — Prétraitement — {CATEGORY}')
print('=' * 60)
print(f'  N_POINTS        : {N_POINTS} points par pièce')
print(f'  Normalisation   : centrage + sphère unitaire ✅')
print(f'  Échantillonnage : random sampling ✅')
print(f'  Augmentations   : rotation Z + jitter + scale ✅')
print(f'  Train loader    : {len(train_loader)} batchs × {BATCH_SIZE}')
print(f'  Val loader      : {len(val_loader)} batchs')
print(f'  Test loader     : {len(test_loader)} batchs')
print(f'  Format tenseur  : (batch, {N_POINTS}, 3) float32 ✅')
print(f'  NaN / Inf       : aucun détecté ✅')
print()
print('  PROCHAINES ÉTAPES (Phase 3 — Modélisation) :')
print('  1. Définir l\'architecture Autoencodeur (encoder PointNet + decoder)')
print('  2. Définir la loss : Chamfer Distance ou Earth Mover Distance')
print('  3. Entraîner sur train (normaux uniquement)')
print('  4. Score d\'anomalie = erreur de reconstruction point à point')
print('  5. Seuil de détection calibré sur la validation')
print('=' * 60)